# Notebook 01 — Data Cleaning, Target Construction & Train/Test Split

**Proyecto:** Boston Marathon BQ Predictor  
**Autor:** Gian Marco  
**Fecha:** Abril 2026

## Objetivos

1. Cargar los 3 CSVs (Results, Races, BQStandards)
2. Aplicar criterios de limpieza documentados en `DECISIONS.md`
3. Construir la variable target `es_BQ`
4. Hacer merge con metadata de carreras (Races.csv)
5. Generar muestra estratificada de 300k
6. Aplicar split temporal: train 2022-2023 / test 2024
7. Guardar datasets procesados

## Referencia

Todas las decisiones aplicadas aquí están justificadas en `DECISIONS.md`.

---
## 1. Imports y configuración

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Reproducibilidad
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Rutas
RAW_DATA_DIR = Path('../data/raw')
PROCESSED_DATA_DIR = Path('../data/processed')
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Config pandas para ver todo
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

---
## 2. Carga de los 3 datasets

In [ ]:
results = pd.read_csv(RAW_DATA_DIR / 'Results.csv', low_memory=False)
races = pd.read_csv(RAW_DATA_DIR / 'Races.csv')
bq = pd.read_csv(RAW_DATA_DIR / 'BQStandards.csv')

print(f'Results: {results.shape}')
print(f'Races:   {races.shape}')
print(f'BQ:      {bq.shape}')

In [ ]:
# Vista rápida
results.head(3)

In [ ]:
races.head(3)

In [ ]:
bq.head(3)

---
## 3. Limpieza de Results.csv

Aplicamos los filtros documentados en `DECISIONS.md` Decisión 6. Llevamos el conteo de cuántos registros eliminamos en cada paso para tener trazabilidad.

In [ ]:
df = results.copy()
cleaning_log = [('Inicial', len(df))]

# 3.1 Age a numérico (había valores con espacio en blanco)
df['Age'] = pd.to_numeric(df['Age'], errors='coerce')

# 3.2 Quitar DNF (Finish = 0)
df = df[df['Finish'] > 0]
cleaning_log.append(('Tras quitar DNF', len(df)))

# 3.3 Solo M y F (BQ Standards no tiene definidos otros géneros)
df = df[df['Gender'].isin(['M', 'F'])]
cleaning_log.append(('Tras filtrar M/F', len(df)))

# 3.4 Edades razonables (18-85)
df = df[(df['Age'] >= 18) & (df['Age'] <= 85)]
cleaning_log.append(('Tras filtrar edad válida', len(df)))

# 3.5 Drop columnas con muchos nulos o sin valor predictivo
df = df.drop(columns=['Zip', 'City', 'State', 'Name'])
cleaning_log.append(('Tras drop columnas irrelevantes', len(df)))

# Log visual
print('LOG DE LIMPIEZA DE RESULTS')
print('=' * 50)
for paso, n in cleaning_log:
    print(f'{paso:40s} {n:>12,}')

### 3.6 Normalizar Country

Detectamos duplicados en códigos de país (`US`/`USA`, `GB`/`GBR`, `DE`/`GER`). Los unificamos.

In [ ]:
country_mapping = {
    'USA': 'US',
    'GBR': 'GB',
    'GER': 'DE',
}
df['Country'] = df['Country'].replace(country_mapping)
df['Country'] = df['Country'].fillna('UNKNOWN')

print(f'Países únicos tras normalizar: {df["Country"].nunique()}')
print(f'\nTop 10:')
print(df['Country'].value_counts().head(10))

---
## 4. Merge con Races.csv

Queremos traernos info de la carrera: `Category` (dificultad), `Finishers` (tamaño), y filtrar solo `Include == 'Yes'` para respetar el filtrado del autor.

⚠️ **Nota importante:** Antes del filtro `Include=Yes`, guardamos un snapshot con todas las carreras para poder construir el slice narrativo de España más adelante (las carreras españolas no tienen `Include=Yes` porque el autor se enfocó en USA/Canadá + London/Berlin).

In [ ]:
# Guardar snapshot con todas las carreras (pre-filtro Include) para el slice España
df_all_races = df.copy()

# Ahora sí filtramos para el pipeline principal
races_clean = races[races['Include'] == 'Yes'].copy()
races_clean = races_clean[['Year', 'Race', 'City', 'State', 'Finishers', 'Category']]
races_clean = races_clean.rename(columns={'City': 'Race_City', 'State': 'Race_State'})

n_before = len(df)
df = df.merge(races_clean, on=['Year', 'Race'], how='inner')
print(f'Filas tras merge (solo carreras Include=Yes): {len(df):,}')
print(f'Filas eliminadas: {n_before - len(df):,}')

cleaning_log.append(('Tras merge con Races (Include=Yes)', len(df)))

---
## 5. Construcción del target `es_BQ`

Asignamos a cada corredor su franja de edad según BQ Standards, hacemos merge y creamos la variable target.

In [ ]:
def assign_age_bracket(age):
    """Asigna la franja de edad según categorías oficiales BQ."""
    if age < 35: return 'Under 35'
    elif age < 40: return '35-39'
    elif age < 45: return '40-44'
    elif age < 50: return '45-49'
    elif age < 55: return '50-54'
    elif age < 60: return '55-59'
    elif age < 65: return '60-64'
    elif age < 70: return '65-69'
    elif age < 75: return '70-74'
    elif age < 80: return '75-79'
    else: return '80 and Over'

df['Age Bracket'] = df['Age'].apply(assign_age_bracket)

# Merge con BQ para traer Standard
df = df.merge(bq, on=['Gender', 'Age Bracket'], how='left')

# Crear target
df['es_BQ'] = (df['Finish'] <= df['Standard']).astype(int)

print('Distribución del target:')
print(df['es_BQ'].value_counts())
print(f'\n% BQ: {df["es_BQ"].mean()*100:.2f}%')
print(f'Ratio desbalance: 1:{(df["es_BQ"]==0).sum() / (df["es_BQ"]==1).sum():.2f}')

---
## 6. Sanity checks

Antes de muestrear, verificamos que los datos tienen sentido.

In [ ]:
print('% BQ por año (debe ir creciendo según nuestra observación previa):')
print(df.groupby('Year')['es_BQ'].agg(['mean', 'sum', 'count']).round(3))

print('\n% BQ por género:')
print(df.groupby('Gender')['es_BQ'].agg(['mean', 'count']).round(3))

print('\nDistribución de carreras (top 10):')
print(df['Race'].value_counts().head(10))

---
## 7. Muestreo estratificado

**Estrategia:** Muestrear 300k filas estratificando por `es_BQ × Year × Gender` para preservar las distribuciones críticas (ver `DECISIONS.md` Decisión 4).

In [ ]:
TARGET_SAMPLE_SIZE = 300_000

# Calcular la fracción
frac = TARGET_SAMPLE_SIZE / len(df)
print(f'Fracción a muestrear: {frac:.4f}')

# Estratificación por múltiples columnas combinadas
strata = df['es_BQ'].astype(str) + '_' + df['Year'].astype(str) + '_' + df['Gender']

df_sample = (
    df.groupby(strata, group_keys=False)
      .apply(lambda g: g.sample(frac=frac, random_state=RANDOM_STATE))
      .reset_index(drop=True)
)

print(f'Tamaño de la muestra: {len(df_sample):,}')
print(f'\n% BQ en muestra: {df_sample["es_BQ"].mean()*100:.2f}% (objetivo: ~12.76%)')

print('\n% BQ por año en muestra (debe parecerse al original):')
print(df_sample.groupby('Year')['es_BQ'].agg(['mean', 'count']).round(3))

---
## 8. Split temporal

**Train:** 2022-2023
**Test:** 2024

Justificación completa en `DECISIONS.md` Decisión 3.

In [ ]:
train = df_sample[df_sample['Year'].isin([2022, 2023])].reset_index(drop=True)
test = df_sample[df_sample['Year'] == 2024].reset_index(drop=True)

print(f'Train: {len(train):,} filas  |  % BQ: {train["es_BQ"].mean()*100:.2f}%')
print(f'Test:  {len(test):,} filas  |  % BQ: {test["es_BQ"].mean()*100:.2f}%')
print(f'\nRatio train/test: {len(train)/len(test):.2f}')

---
## 9. Slice narrativo: España

Reservamos los registros de carreras españolas para análisis posterior (notebook 09) y la demo de Streamlit.

In [ ]:
# Aplicamos el mismo proceso de target a df_all_races (pre-filtro Include)
df_all_races['Age Bracket'] = df_all_races['Age'].apply(assign_age_bracket)
df_all_races = df_all_races.merge(bq, on=['Gender', 'Age Bracket'], how='left')
df_all_races['es_BQ'] = (df_all_races['Finish'] <= df_all_races['Standard']).astype(int)

# Identificar carreras celebradas en España
spain_keywords = ['Madrid', 'Barcelona', 'Sevilla', 'Valencia', 'Lanzarote']
spain_races_mask = df_all_races['Race'].str.contains('|'.join(spain_keywords), case=False, na=False)

# 'Zurich Marato' (catalán/español) para Barcelona/Sevilla
zurich_spain = df_all_races['Race'].str.contains('Zurich Marato', case=False, na=False)
spain_races_mask = spain_races_mask | zurich_spain

spain_slice = df_all_races[spain_races_mask].reset_index(drop=True)

print(f'Slice España: {len(spain_slice):,} filas')
print(f'\nCarreras españolas incluidas:')
print(spain_slice.groupby(['Race', 'Year']).agg(
    finishers=('es_BQ', 'count'),
    bq_pct=('es_BQ', 'mean')
).round(3))

---
## 10. Guardar datasets procesados

In [ ]:
train.to_csv(PROCESSED_DATA_DIR / 'train.csv', index=False)
test.to_csv(PROCESSED_DATA_DIR / 'test.csv', index=False)
spain_slice.to_csv(PROCESSED_DATA_DIR / 'spain_slice.csv', index=False)

# Guardar también el log de limpieza para documentación
cleaning_df = pd.DataFrame(cleaning_log, columns=['Paso', 'N_filas'])
cleaning_df.to_csv(PROCESSED_DATA_DIR / 'cleaning_log.csv', index=False)

print('✅ Archivos guardados en data/processed/:')
for f in sorted(PROCESSED_DATA_DIR.iterdir()):
    size_mb = f.stat().st_size / 1024**2
    print(f'  {f.name:25s} {size_mb:>6.1f} MB')

---
## Resumen del notebook

| Paso | Resultado |
|---|---|
| Datos originales | 1.76M filas |
| Tras limpieza | ~1.6M filas |
| Tras merge con Races (Include=Yes) | ~1.5M filas (estimado) |
| Muestra estratificada | 300k filas |
| Train (2022-2023) | ~265k filas |
| Test (2024) | ~35k filas |
| Slice España | ~6k filas |
| **Target % BQ** | **~12.76%** |